In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [3]:
!wget -c https://dumps.wikimedia.org/fawiki/latest/fawiki-latest-pages-articles-multistream.xml.bz2

--2026-07-26 20:43:53--  https://dumps.wikimedia.org/fawiki/latest/fawiki-latest-pages-articles-multistream.xml.bz2
Resolving dumps.wikimedia.org (dumps.wikimedia.org)... 208.80.154.242, 2620:0:861:ed1a::3:242
Connecting to dumps.wikimedia.org (dumps.wikimedia.org)|208.80.154.242|:443... connected.
HTTP request sent, awaiting response... 416 Requested Range Not Satisfiable

    The file is already fully retrieved; nothing to do.



In [4]:
DUMP_PATH = "/kaggle/working/fawiki-latest-pages-articles-multistream.xml.bz2"


In [5]:
import bz2
import mwxml
from tqdm.auto import tqdm

dump = mwxml.Dump.from_file(bz2.open(DUMP_PATH, "rb"))

count = 0

for page in dump:
    print("Page ID:", page.id)
    print("Title:", page.title)
    print("Namespace:", page.namespace)
    print("Redirect:", page.redirect)
    print("-" * 50)

    count += 1
    if count >= 10:
        break


Page ID: 2
Title: صفحهٔ اصلی
Namespace: 0
Redirect: None
--------------------------------------------------
Page ID: 10
Title: مدیاویکی:Category
Namespace: 8
Redirect: None
--------------------------------------------------
Page ID: 22
Title: مدیاویکی:Bugreports
Namespace: 8
Redirect: None
--------------------------------------------------
Page ID: 23
Title: مدیاویکی:Bugreportspage
Namespace: 8
Redirect: None
--------------------------------------------------
Page ID: 67
Title: مدیاویکی:Lastmodified
Namespace: 8
Redirect: None
--------------------------------------------------
Page ID: 74
Title: مدیاویکی:Sysoptext
Namespace: 8
Redirect: None
--------------------------------------------------
Page ID: 76
Title: مدیاویکی:Developertext
Namespace: 8
Redirect: None
--------------------------------------------------
Page ID: 80
Title: مدیاویکی:Sitesubtitle
Namespace: 8
Redirect: None
--------------------------------------------------
Page ID: 98
Title: مدیاویکی:Noconnect
Namespace: 8
Redirec

In [6]:
def normalize_title(title):
    if title is None:
        return None

    title = str(title).strip()

    if not title:
        return None

    title = title.replace("_", " ")

    if "#" in title:
        title = title.split("#", 1)[0].strip()

    if not title:
        return None

    return title


In [7]:
import mwparserfromhell

def extract_page_data(title, text):
    wikicode = mwparserfromhell.parse(text)

    links = set()
    categories = set()

    for link in wikicode.filter_wikilinks():
        target = normalize_title(link.title)

        if target is None:
            continue

        if target.startswith("رده:"):
            category = target.replace("رده:", "", 1).strip()
            if category:
                categories.add(category)
            continue

        if ":" in target:
            continue

        links.add(target)

    plain_text = wikicode.strip_code().strip()

    return {
        "title": title,
        "plain_text": plain_text,
        "links": sorted(links),
        "categories": sorted(categories)
    }


In [8]:
import mwparserfromhell

def extract_page_data(title, text):
    wikicode = mwparserfromhell.parse(text)

    links = set()
    categories = set()

    for link in wikicode.filter_wikilinks():
        target = normalize_title(link.title)

        if target is None:
            continue

        if target.startswith("رده:"):
            category = target.replace("رده:", "", 1).strip()
            if category:
                categories.add(category)
            continue

        if ":" in target:
            continue

        links.add(target)

    plain_text = wikicode.strip_code().strip()

    return {
        "title": title,
        "plain_text": plain_text,
        "links": sorted(links),
        "categories": sorted(categories)
    }


In [9]:
import pandas as pd

pages = []
link_edges = []
category_edges = []
redirect_edges = []

dump = mwxml.Dump.from_file(bz2.open(DUMP_PATH, "rb"))

max_pages = 1000

for page in tqdm(dump, total=max_pages):
    if len(pages) >= max_pages:
        break

    if page.namespace != 0:
        continue

    title = normalize_title(page.title)

    if title is None:
        continue

    if page.redirect:
        redirect_target = normalize_title(page.redirect)
        if redirect_target:
            redirect_edges.append({
                "source": title,
                "relation": "redirects_to",
                "target": redirect_target
            })
        continue

    revision = next(page, None)

    if revision is None or revision.text is None:
        continue

    data = extract_page_data(title, revision.text)

    pages.append({
        "page_id": page.id,
        "title": title,
        "text": data["plain_text"][:3000],
        "num_links": len(data["links"]),
        "num_categories": len(data["categories"])
    })

    for target in data["links"]:
        link_edges.append({
            "source": title,
            "relation": "links_to",
            "target": target
        })

    for category in data["categories"]:
        category_edges.append({
            "source": title,
            "relation": "belongs_to_category",
            "target": category
        })


  0%|          | 0/1000 [00:00<?, ?it/s]

In [10]:
pages_df = pd.DataFrame(pages)
link_edges_df = pd.DataFrame(link_edges)
category_edges_df = pd.DataFrame(category_edges)
redirect_edges_df = pd.DataFrame(redirect_edges)

print("Pages:", len(pages_df))
print("Link edges:", len(link_edges_df))
print("Category edges:", len(category_edges_df))
print("Redirect edges:", len(redirect_edges_df))


Pages: 1000
Link edges: 94192
Category edges: 8081
Redirect edges: 281


In [11]:
pages_df.head()


,page_id,title,text,num_links,num_categories
0,2,صفحهٔ اصلی,15px|alt=|پیوند= مقاله‌های برگزیده – مقالهٔ ام...,10,0
1,594,ویکی‌پدیا,ویکی‌پدیا ، یک دانشنامه برخط چندزبانه بر پایهٔ...,185,21
2,613,سالنامه,بندانگشتی|جلد یک سالنامه\nسالنامه‌ها یا وقایع‌...,19,4
3,619,اطلاعات,بندانگشتی|منابع اِطّلاعات، اَزدایش\nاِطّلاعات،...,23,8
4,637,محتوای آزاد,محتوای آزاد یا اطلاعات آزاد یا محتوای باز هر ا...,54,6


In [12]:
link_edges_df.head()


,source,relation,target
0,صفحهٔ اصلی,links_to,"Y ""(میلادی)"""
1,صفحهٔ اصلی,links_to,j F
2,صفحهٔ اصلی,links_to,xiY (خورشیدی)
3,صفحهٔ اصلی,links_to,xij xiF
4,صفحهٔ اصلی,links_to,xmY (قمری)


In [13]:
category_edges_df.head()


,source,relation,target
0,ویکی‌پدیا,belongs_to_category,اختراع‌های آمریکایی
1,ویکی‌پدیا,belongs_to_category,اطلاعات منبع‌باز
2,ویکی‌پدیا,belongs_to_category,انقلاب علمی
3,ویکی‌پدیا,belongs_to_category,برندگان جایزه اراسموس
4,ویکی‌پدیا,belongs_to_category,بنیان‌گذاری‌های ۲۰۰۱ (میلادی) در ایالات متحده ...


In [14]:
redirect_edges_df.head()


,source,relation,target
0,دائرةالمعارف,redirects_to,دانشنامه
1,نرم‌افزارهای آزاد,redirects_to,نرم‌افزار آزاد
2,قانون اساسی ایران,redirects_to,قانون اساسی جمهوری اسلامی ایران
3,فهرست کشورها,redirects_to,فهرست کشورهای مستقل
4,انیشتین,redirects_to,آلبرت اینشتین


In [15]:
edges_df = pd.concat(
    [link_edges_df, category_edges_df, redirect_edges_df],
    ignore_index=True
)

edges_df = edges_df.drop_duplicates()

print(edges_df.shape)
edges_df.head()


(102554, 3)


,source,relation,target
0,صفحهٔ اصلی,links_to,"Y ""(میلادی)"""
1,صفحهٔ اصلی,links_to,j F
2,صفحهٔ اصلی,links_to,xiY (خورشیدی)
3,صفحهٔ اصلی,links_to,xij xiF
4,صفحهٔ اصلی,links_to,xmY (قمری)


In [16]:
pages_df.to_parquet("/kaggle/working/fawiki_pages_sample.parquet", index=False)
edges_df.to_parquet("/kaggle/working/fawiki_edges_sample.parquet", index=False)


In [17]:
def get_outgoing_edges(title, edges_df, limit=20):
    result = edges_df[edges_df["source"] == title]
    return result.head(limit)

get_outgoing_edges("انیشتین", edges_df, limit=20)


,source,relation,target
102277,انیشتین,redirects_to,آلبرت اینشتین


In [18]:
from pathlib import Path

DUMP_PATH = Path(
    "/kaggle/working/fawiki-latest-pages-articles-multistream.xml.bz2"
)

OUTPUT_DIR = Path("/kaggle/working/fawiki_graph")

PAGES_DIR = OUTPUT_DIR / "pages"
EDGES_DIR = OUTPUT_DIR / "edges"

PAGES_DIR.mkdir(parents=True, exist_ok=True)
EDGES_DIR.mkdir(parents=True, exist_ok=True)

print(DUMP_PATH.exists())
print(OUTPUT_DIR)


True
/kaggle/working/fawiki_graph


In [19]:
import re
import mwparserfromhell

SPACE_PATTERN = re.compile(r"\s+")

BLOCKED_NAMESPACES = {
    "پرونده",
    "تصویر",
    "File",
    "Image",
    "الگو",
    "Template",
    "بحث",
    "Talk",
    "کاربر",
    "User",
    "ویکی‌پدیا",
    "Wikipedia",
    "راهنما",
    "Help",
    "درگاه",
    "Portal",
    "مدیاویکی",
    "MediaWiki",
    "ویژه",
    "Special",
    "پودمان",
    "Module",
    "پیش‌نویس",
    "Draft"
}

CATEGORY_PREFIXES = {
    "رده",
    "Category"
}


In [20]:
def normalize_title(title):
    if title is None:
        return None

    title = str(title).replace("_", " ").strip()
    title = SPACE_PATTERN.sub(" ", title)

    if title.startswith(":"):
        title = title[1:].strip()

    if "#" in title:
        title = title.split("#", 1)[0].strip()

    return title or None


In [21]:
def split_namespace(title):
    if ":" not in title:
        return None, title

    prefix, remainder = title.split(":", 1)

    return prefix.strip(), remainder.strip()


In [22]:
def extract_page_content(text):
    if not text:
        return "", [], []

    wikicode = mwparserfromhell.parse(text)

    links = set()
    categories = set()

    for wikilink in wikicode.filter_wikilinks(recursive=True):
        target = normalize_title(wikilink.title)

        if not target:
            continue

        prefix, remainder = split_namespace(target)

        if prefix in CATEGORY_PREFIXES:
            category = normalize_title(remainder)

            if category:
                categories.add(category)

            continue

        if prefix in BLOCKED_NAMESPACES:
            continue

        links.add(target)

    plain_text = wikicode.strip_code(
        normalize=True,
        collapse=True
    ).strip()

    return plain_text, sorted(links), sorted(categories)


# Revision Function

In [23]:
def get_last_revision(page):
    last_revision = None

    for revision in page:
        last_revision = revision

    return last_revision


In [24]:
PAGE_COLUMNS = [
    "page_id",
    "title",
    "namespace",
    "is_redirect",
    "redirect_target",
    "text",
    "num_links",
    "num_categories"
]

EDGE_COLUMNS = [
    "source",
    "relation",
    "target"
]


In [25]:
def save_batch(pages, edges, batch_index):
    pages_path = PAGES_DIR / f"pages_{batch_index:05d}.parquet"
    edges_path = EDGES_DIR / f"edges_{batch_index:05d}.parquet"

    pages_df = pd.DataFrame(pages, columns=PAGE_COLUMNS)

    edges_df = pd.DataFrame(edges, columns=EDGE_COLUMNS)
    edges_df = edges_df.drop_duplicates()

    pages_df.to_parquet(
        pages_path,
        index=False,
        compression="snappy"
    )

    edges_df.to_parquet(
        edges_path,
        index=False,
        compression="snappy"
    )

    return len(pages_df), len(edges_df)


In [26]:
import bz2
import gc
import json
import time
import mwxml

from tqdm.auto import tqdm


In [27]:
BATCH_SIZE = 5000


In [28]:
!rm -rf /kaggle/working/fawiki_graph/pages/*
!rm -rf /kaggle/working/fawiki_graph/edges/*


In [ ]:
pages_buffer = []
edges_buffer = []

batch_index = 0
total_pages = 0
total_edges = 0
total_redirects = 0
failed_pages = 0

start_time = time.time()

with bz2.open(DUMP_PATH, "rb") as dump_file:
    dump = mwxml.Dump.from_file(dump_file)

    progress = tqdm(desc="Processed main-namespace pages", unit="page")

    for page in dump:
        if page.namespace != 0:
            continue

        title = normalize_title(page.title)

        if not title:
            continue

        redirect_target = normalize_title(page.redirect)
        is_redirect = redirect_target is not None

        if is_redirect:
            pages_buffer.append({
                "page_id": page.id,
                "title": title,
                "namespace": page.namespace,
                "is_redirect": True,
                "redirect_target": redirect_target,
                "text": "",
                "num_links": 0,
                "num_categories": 0
            })

            edges_buffer.append({
                "source": title,
                "relation": "redirects_to",
                "target": redirect_target
            })

            total_redirects += 1

        else:
            try:
                revision = get_last_revision(page)

                if revision is None:
                    text = ""
                else:
                    text = revision.text or ""

                plain_text, links, categories = extract_page_content(text)

                pages_buffer.append({
                    "page_id": page.id,
                    "title": title,
                    "namespace": page.namespace,
                    "is_redirect": False,
                    "redirect_target": None,
                    "text": plain_text,
                    "num_links": len(links),
                    "num_categories": len(categories)
                })

                for target in links:
                    if target == title:
                        continue

                    edges_buffer.append({
                        "source": title,
                        "relation": "links_to",
                        "target": target
                    })

                for category in categories:
                    edges_buffer.append({
                        "source": title,
                        "relation": "belongs_to_category",
                        "target": category
                    })

            except Exception as error:
                failed_pages += 1

                if failed_pages <= 20:
                    print(f"Failed page: {title} | {error}")

        total_pages += 1
        progress.update(1)

        if len(pages_buffer) >= BATCH_SIZE:
            saved_pages, saved_edges = save_batch(
                pages_buffer,
                edges_buffer,
                batch_index
            )

            total_edges += saved_edges

            elapsed_minutes = (time.time() - start_time) / 60

            progress.set_postfix({
                "batch": batch_index,
                "pages": total_pages,
                "edges": total_edges,
                "failed": failed_pages,
                "minutes": round(elapsed_minutes, 1)
            })

            pages_buffer.clear()
            edges_buffer.clear()

            batch_index += 1

            gc.collect()

    progress.close()


Processed main-namespace pages: 0page [00:00, ?page/s]

# doing last batch 

In [ ]:
if pages_buffer:
    saved_pages, saved_edges = save_batch(
        pages_buffer,
        edges_buffer,
        batch_index
    )

    total_edges += saved_edges
    batch_index += 1

pages_buffer.clear()
edges_buffer.clear()

gc.collect()


# save report

In [ ]:
elapsed_seconds = time.time() - start_time

metadata = {
    "dump_path": str(DUMP_PATH),
    "batch_size": BATCH_SIZE,
    "number_of_batches": batch_index,
    "total_pages": total_pages,
    "total_edges_after_local_deduplication": total_edges,
    "total_redirects": total_redirects,
    "failed_pages": failed_pages,
    "elapsed_seconds": elapsed_seconds,
    "elapsed_minutes": elapsed_seconds / 60
}

metadata_path = OUTPUT_DIR / "metadata.json"

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(
        metadata,
        file,
        ensure_ascii=False,
        indent=2
    )

print(json.dumps(metadata, ensure_ascii=False, indent=2))


# test what we have

In [ ]:
page_files = sorted(PAGES_DIR.glob("*.parquet"))
edge_files = sorted(EDGES_DIR.glob("*.parquet"))

print("Page files:", len(page_files))
print("Edge files:", len(edge_files))
print("First page file:", page_files[0] if page_files else None)
print("First edge file:", edge_files[0] if edge_files else None)


# reading sample 

In [ ]:
sample_pages_df = pd.read_parquet(page_files[0])
sample_edges_df = pd.read_parquet(edge_files[0])

display(sample_pages_df.head())
display(sample_edges_df.head())


In [ ]:
print(sample_pages_df.columns.tolist())
print(sample_edges_df.columns.tolist())


# Save output

In [ ]:
from pathlib import Path
import json
import shutil
import time
import os

OUTPUT_DIR = Path("/kaggle/working/fawiki_graph")
PAGES_DIR = OUTPUT_DIR / "pages"
EDGES_DIR = OUTPUT_DIR / "edges"

FINAL_REPORT_PATH = OUTPUT_DIR / "final_report.json"
ZIP_BASE_PATH = Path("/kaggle/working/fawiki_graph_backup")
ZIP_PATH = Path("/kaggle/working/fawiki_graph_backup.zip")

assert OUTPUT_DIR.exists(), f"Output directory does not exist: {OUTPUT_DIR}"
assert PAGES_DIR.exists(), f"Pages directory does not exist: {PAGES_DIR}"
assert EDGES_DIR.exists(), f"Edges directory does not exist: {EDGES_DIR}"

page_files = sorted(PAGES_DIR.glob("*.parquet"))
edge_files = sorted(EDGES_DIR.glob("*.parquet"))

assert len(page_files) > 0, "No page parquet files found."
assert len(edge_files) > 0, "No edge parquet files found."

def get_directory_size_bytes(path):
    total_size = 0

    for root, dirs, files in os.walk(path):
        for file_name in files:
            file_path = Path(root) / file_name
            if file_path.exists():
                total_size += file_path.stat().st_size

    return total_size

output_size_bytes = get_directory_size_bytes(OUTPUT_DIR)

final_report = {
    "status": "completed",
    "output_dir": str(OUTPUT_DIR),
    "pages_dir": str(PAGES_DIR),
    "edges_dir": str(EDGES_DIR),
    "num_page_files": len(page_files),
    "num_edge_files": len(edge_files),
    "output_size_bytes": output_size_bytes,
    "output_size_gb": round(output_size_bytes / (1024 ** 3), 4),
    "created_at_unix": time.time(),
    "important_note": "Files are stored under /kaggle/working. Use Save Version / Commit in Kaggle to preserve them in Notebook Output."
}

with open(FINAL_REPORT_PATH, "w", encoding="utf-8") as file:
    json.dump(final_report, file, ensure_ascii=False, indent=2)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

shutil.make_archive(
    base_name=str(ZIP_BASE_PATH),
    format="zip",
    root_dir=str(OUTPUT_DIR)
)

zip_size_bytes = ZIP_PATH.stat().st_size

final_report["zip_path"] = str(ZIP_PATH)
final_report["zip_size_bytes"] = zip_size_bytes
final_report["zip_size_gb"] = round(zip_size_bytes / (1024 ** 3), 4)

with open(FINAL_REPORT_PATH, "w", encoding="utf-8") as file:
    json.dump(final_report, file, ensure_ascii=False, indent=2)

print("Final output is ready.")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Page parquet files: {len(page_files)}")
print(f"Edge parquet files: {len(edge_files)}")
print(f"Output size: {final_report['output_size_gb']} GB")
print(f"Backup zip: {ZIP_PATH}")
print(f"Zip size: {final_report['zip_size_gb']} GB")
print()
print("Now click Save Version / Commit in Kaggle.")
print("After saving, check the Notebook Output section for:")
print(f"- {OUTPUT_DIR}")
print(f"- {ZIP_PATH}")
